In [ ]:
# https://jumin.mois.go.kr/#
import pandas as pd

df = pd.read_csv(
    "../../data/202012_202512_주민등록인구및세대현황_연간.csv", encoding="cp949"
)

df

In [ ]:
df.loc[[10, 11, 14, 15], :]

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.describe()

In [ ]:
df["행정구역"] = df["행정구역"].str.split("(").str[0].str.strip()  # 이름만 추출

name_mapping = {"강원도": "강원특별자치도", "전라북도": "전북특별자치도"}

df["행정구역"] = df["행정구역"].replace(name_mapping)

df_combined = df.groupby("행정구역").sum().reset_index()
df_combined.head()


In [ ]:
df_combined.describe()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# 1. 데이터 준비 (남자인구수, 여자인구수 컬럼만 추출)
# 컬럼 이름이 '2020년_남자인구수', '2020년_여자인구수' 형태라고 가정합니다.
male_cols = [col for col in df_combined.columns if "남자 인구수" in col]
female_cols = [col for col in df_combined.columns if "여자 인구수" in col]

# # 2. 연도별 전체 합계 계산
# # 모든 지역(행)의 값을 더해 연도별 총 남여 인구수를 구합니다.
# male_sum = df[male_cols].sum()
# female_sum = df[female_cols].sum()

# # 3. 시각화를 위한 데이터프레임 재구성
# years = [col.split("년")[0] for col in male_cols]  # '2020', '2021' 등 연도만 추출

# plot_df = pd.DataFrame(
#     {
#         "연도": years * 2,
#         "인구수": list(male_sum) + list(female_sum),
#         "성별": ["남자"] * len(years) + ["여자"] * len(years),
#     }
# )

# # 4. 선 그래프 그리기
# plt.figure(figsize=(12, 6))

# # sns.lineplot을 사용하여 남자와 여자 선을 따로 그립니다.
# sns.lineplot(data=plot_df, x="연도", y="인구수", hue="성별", marker="o", linewidth=2.5)

# # 그래프 꾸미기
# plt.title("연도별 남여 인구수 변화 추이", fontsize=15, pad=20)
# plt.xlabel("연도", fontsize=12)
# plt.ylabel("인구수 (명)", fontsize=12)
# plt.grid(True, linestyle="--", alpha=0.6)
# plt.legend(title="성별")

# # y축 천단위 콤마 표시 (인구수가 클 경우 대비)
# current_values = plt.gca().get_yticks()
# plt.gca().set_yticklabels(["{:,.0f}".format(x) for x in current_values])

# plt.show()

In [ ]:
plt.rc("font", family="Malgun Gothic")

plt.figure(figsize=(10, 6))
avg_population = [2.23, 2.18, 2.15, 2.13, 2.10, 2.08]
years = ["2020", "2021", "2022", "2023", "2024", "2025"]

sns.lineplot(x=years, y=avg_population, marker="o", linewidth=2)
plt.title("연도별 평균 세대당 인구 변화")
plt.ylabel("인구 수 (명)")
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# '세대당 인구' 컬럼들만 필터링 (시각화용)
pop_cols = [col for col in df_combined if "세대당 인구" in col]
heatmap_df = df_combined.set_index("행정구역")[pop_cols]

# 컬럼명을 연도만 나오게 짧게 변경 (예: 2020년_세대당 인구 -> 2020)
heatmap_df.columns = [col.split("_")[0] for col in heatmap_df.columns]

# --- 시각화 시작 ---

# 1. 히트맵 (Heatmap)
# 지역별/연도별 세대당 인구의 밀도를 확인
plt.figure(figsize=(12, 8))
sns.heatmap(
    heatmap_df,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    cbar_kws={"label": "세대당 인구(명)"},
)
plt.title("지역별 세대당 인구 변화 추이 (Heatmap)")
plt.show()

# 2. 박스 플롯 (Box Plot)
# 연도별로 데이터가 얼마나 퍼져있는지(분포) 확인
# 데이터 구조를 가로에서 세로로 변경 (Tidy Data)
df_long = heatmap_df.melt(var_name="연도", value_name="세대당 인구")

plt.figure(figsize=(10, 6))
sns.boxplot(
    x="연도", y="세대당 인구", data=df_long, palette="vlag", hue="연도", legend=False
)
sns.stripplot(
    x="연도", y="세대당 인구", data=df_long, color="black", alpha=0.3
)  # 실제 점들도 표시
plt.title("연도별 세대당 인구 분포 변화 (Boxplot)")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()